This is a jupyter notebook; you can open with Jupyter or run it on google colab. It is a frozen snapshot of a live script you can interact with/change/rerun on the fly thru the notebook interface.
```sh
jupyter notebook research/notebooks/DistanceCalc.ipynb
```
or test with a different python version:
```sh
uv run --python=3.13 jupyter notebook  research/notebooks/DistanceCalc.ipynb
```

---

We calculate euclidean distances using the Pythagorean formula: `d = root ( x^2 + y^2 + z^2 )`.

When you don't *need* the value, only to test against another value, such as to say "less than R", it is faster to square R and test the un-rooted value:

```
d = x^2 + y^2 + z^2
return d < r
```

This is code I used to research the optimal algorithm for calculating the distance component.

Calculating squares: is it better to calculate (x1-x2)**2 (squared), to perform the subtraction twice (x1-x2)*(x1-x2), or to store the subtractions into temporarys (this has an overhead) (dx = x1-x2, d = dx*dx + dy*dy...)
Calculating roots: during the Python 2->3 transition and 32-64bit transitions, math.sqrt was sometimes slower than operator `**0.5`. This no-longer appears to be true, unless you use it as `math.sqrt` in which case the overhead of a global-namespace module lookup on `math` and a method lookup `sqrt` at every call gives `**` an edge.

Since `math.sqrt` means a lookup on `math` to find `sqrt` for every call, let's remove *one* lookup

In [1]:
from math import sqrt as msqrt

Simulate the `System` class with a simple point type.

In [2]:
class P:
    def __init__(self, x, y, z):
        self.posX, self.posY, self.posZ = x, y, z

    # Direct implementations of the algorithm with no temporary variables
    def fn1(self, p2):
        return msqrt( (self.posX - p2.posX)**2 + (self.posY - p2.posY)**2 + (self.posZ - p2.posZ)**2 )
    def fn2(self, p2):
        return msqrt( (self.posX - p2.posX)*(self.posX - p2.posX) + (self.posY - p2.posY)*(self.posY - p2.posY) + (self.posZ - p2.posZ)*(self.posZ - p2.posZ) )
    def fn3(self, p2):
        return ( (self.posX - p2.posX)**2 + (self.posY - p2.posY)**2 + (self.posZ - p2.posZ)**2 ) ** 0.5
    def fn4(self, p2):
        return ( (self.posX - p2.posX)*(self.posX - p2.posX) + (self.posY - p2.posY)*(self.posY - p2.posY) + (self.posZ - p2.posZ)*(self.posZ - p2.posZ) ) ** 0.5

    # I'm fairly sure **2 will be expensive, and taking temporaries might be faster,
    # especially post-jit.
    def fn5(self, p2):
        dx, dy, dz = self.posX - p2.posX, self.posY - p2.posY, self.posZ - p2.posZ
        return msqrt( dx*dx + dy*dy + dz*dz )
    def fn6(self, p2):
        dx, dy, dz = self.posX - p2.posX, self.posY - p2.posY, self.posZ - p2.posZ
        return ( dx*dx + dy*dy + dz*dz ) ** 0.5

Free-standing methods to eliminate the per-call method lookup.

In [3]:
def sysdist1(p1, p2):
    return msqrt( (p1.posX - p2.posX)**2 + (p1.posY - p2.posY)**2 + (p1.posZ - p2.posZ)**2 )
    
def sysdist2(p1, p2):
    return ( (p1.posX - p2.posX)**2 + (p1.posY - p2.posY)**2 + (p1.posZ - p2.posZ)**2 ) ** 0.5
    
def sysdist3(p1, p2):
    dx, dy, dz = p1.posX - p2.posX, p1.posY - p2.posY, p1.posZ - p2.posZ
    return msqrt( dx*dx + dy*dy + dz*dz )

def sysdist4(p1, p2):
    dx, dy, dz = p1.posX - p2.posX, p1.posY - p2.posY, p1.posZ - p2.posZ
    return ( dx*dx + dy*dy + dz*dz ) ** 0.5

def sysdist5(p1, p2):
    dx, dy, dz = p1.posX - p2.posX, p1.posY - p2.posY, p1.posZ - p2.posZ
    return msqrt( dx**2 + dy**2 + dz**2 )
    
def sysdist6(p1, p2):
    dx, dy, dz = p1.posX - p2.posX, p1.posY - p2.posY, p1.posZ - p2.posZ
    return ( dx**2 + dy**2 + dz**2 ) ** 0.5

Define some points with a range of values.

In [4]:
p1 = P(3402., 4702., -65535.)
points = [
    p1,
    P(-102.1, 333.1, 44.1), P(23401.1, -9991.5, 4474702893.1), P(0.0, 0.0, 0.0), P(0.1, 0.1, 0.1), P(-0.1, -0.1, -0.1),
    P(-1234., -1234., -1234.), P(23456., 23456., 23456.), P(1., 100., 10000000.),
    P(-999999999., 99999999., 9.0000009)
]


Battery of tests: fn1 = method function, sd1 = stand alone function

In [5]:
print(f"there are {len(points)} points and a loop overhead, so divide each time by {int(len(points) * 1.1)} to get a ~per-call cost")

print("-- msqrt( (self.posX - p2.posX)**2 ...")
%timeit -r 11 -n 500000 for p2 in points: p1.fn1(p2)
%timeit -r 11 -n 500000 for p2 in points: sysdist1(p1, p2)
print()

print("-- msqrt( (self.posX - p2.posX)*(self...)...)")
%timeit -r 11 -n 500000 for p2 in points: p1.fn2(p2)
%timeit -r 11 -n 500000 for p2 in points: sysdist2(p1, p2)
print()

print("-- ( (self.posX - p2.posX)**2 .. ) ** 0.5")
%timeit -r 11 -n 500000 for p2 in points: p1.fn3(p2)
%timeit -r 11 -n 500000 for p2 in points: sysdist3(p1, p2)
print()

print("-- ( (self.posX - p2.posX)*(self...)...) ** 0.5")
%timeit -r 11 -n 500000 for p2 in points: p1.fn4(p2)
%timeit -r 11 -n 500000 for p2 in points: sysdist4(p1, p2)
print()

print("msqrt(dx+dy+dz)")
%timeit -r 11 -n 500000 for p2 in points: p1.fn5(p2)
%timeit -r 11 -n 500000 for p2 in points: sysdist5(p1, p2)
print()

print("(dx+dy+dz)**0.5")
%timeit -r 11 -n 500000 for p2 in points: p1.fn6(p2)
%timeit -r 11 -n 500000 for p2 in points: sysdist6(p1, p2)
print()


there are 10 points and a loop overhead, so divide each time by 11 to get a ~per-call cost
-- msqrt( (self.posX - p2.posX)**2 ...
5.17 μs ± 135 ns per loop (mean ± std. dev. of 11 runs, 500,000 loops each)
5.09 μs ± 136 ns per loop (mean ± std. dev. of 11 runs, 500,000 loops each)

-- msqrt( (self.posX - p2.posX)*(self...)...)
4.05 μs ± 153 ns per loop (mean ± std. dev. of 11 runs, 500,000 loops each)
5.06 μs ± 118 ns per loop (mean ± std. dev. of 11 runs, 500,000 loops each)

-- ( (self.posX - p2.posX)**2 .. ) ** 0.5
5.27 μs ± 113 ns per loop (mean ± std. dev. of 11 runs, 500,000 loops each)
3.23 μs ± 134 ns per loop (mean ± std. dev. of 11 runs, 500,000 loops each)

-- ( (self.posX - p2.posX)*(self...)...) ** 0.5
4.03 μs ± 75.9 ns per loop (mean ± std. dev. of 11 runs, 500,000 loops each)
3.4 μs ± 52.1 ns per loop (mean ± std. dev. of 11 runs, 500,000 loops each)

msqrt(dx+dy+dz)
3.3 μs ± 88 ns per loop (mean ± std. dev. of 11 runs, 500,000 loops each)
5.18 μs ± 83.7 ns per loop (mea

---

A time of 5.12 us above means we spend about 500ns calculating each distance, so calculating a thousand distances = 500us, a million = 0.5s.